# Pull Request Review

A practical guide to reviewing one branch against its target branch. Use this notebook when deciding whether a Pull Request is ready to merge.

## 1. What a Pull Request review answers

A Pull Request (PR) proposes merging one branch into another, for example:

```text
feature/add-search  ->  development
```

A useful review answers four questions:

1. Does the branch solve the intended problem?
2. Is the implementation correct, safe, and maintainable?
3. Does it work with the current target branch?
4. Has the relevant behavior been tested?

A review is more than reading commit messages. Base it on the actual diff, the affected code, and relevant tests.

## 2. Prepare a safe local review

A PR does not need to be approved or merged before you can inspect it locally. First ensure that your own work is safe:

```bash
git status
git fetch origin
git branch -r
```

If the branch exists only on the remote, create a local tracking branch:

```bash
git switch --track origin/feature/add-search
```

You can now inspect and test the proposed changes without merging them into `development`. When finished, return to your previous branch with `git switch <branch-name>`.

## 3. Compare the branch with its target

Use the current remote target as the baseline:

```bash
git fetch origin
git log --oneline origin/development..HEAD
git diff --stat origin/development...HEAD
git diff origin/development...HEAD
```

The three-dot comparison shows the changes introduced by the branch since it split from `development`. Review both the changed files and the surrounding code they depend on.

A visual history can also help:

```bash
git log --oneline --decorate --graph --all
```

## 4. A focused review flow

```text
Read the PR description
        |
Inspect the actual diff
        |
Understand affected code and configuration
        |
Run relevant tests
        |
Leave clear feedback
        |
Approve, comment, or request changes
```

Keep comments specific. Explain the impact, point to the relevant code, and suggest a direction when possible.

## 5. Review checklist

### Scope and design

- Is the purpose of the change clear?
- Does the PR contain only work relevant to that purpose?
- Does the design fit the existing architecture and conventions?
- Are names, interfaces, and documentation clear?

### Correctness and maintainability

- Does the code handle normal and failure cases correctly?
- Are errors useful and handled at an appropriate level?
- Could repeated execution create duplicate data or side effects?
- Are changed paths, imports, and configuration values consistent?

### Configuration and security

- Are environment variables and configuration names consistent?
- Are secrets excluded from Git and from application source code?
- If containers changed, are services, ports, volumes, networks, and build paths correct?
- If dependencies changed, are versions compatible and justified?

### Testing

- Are tests included or updated when appropriate?
- Do automated checks pass?
- What behavior should be verified manually?

## 6. Test the change

Testing should match the purpose of the PR. For example:

```bash
pytest
docker compose config
docker compose up --build
```

Also verify the behavior a user or another service would observe. A passing unit test is useful, but it may not reveal an incorrect configuration, a missing environment variable, or an integration problem.

Do not rely only on the author's claim that the branch was tested. Reproduce the important checks when practical.

## 7. Approve, comment, or request changes

### Approve

Approve when the PR solves its intended problem, relevant tests pass, and no important correctness or security issue remains.

### Comment

Comment when you need clarification or have a non-blocking suggestion.

> Is this fallback behavior intentional when the configuration value is missing?

### Request changes

Request changes when an issue should be fixed before merging, for example a broken feature, committed secret, missing required behavior, unsafe data handling, or incompatible configuration.

## 8. Prompt example for a structured review

> Review the current branch as a Pull Request against `origin/development`.
>
> Do not modify any files. Base the review on the actual diff between the current branch and `origin/development`.
>
> Summarize: the purpose of the changes, affected files, and their impact on the existing system.
>
> Review for: correctness, scope, maintainability, configuration, secrets, changed paths or volumes, dependencies, error handling, duplicate processing, and missing tests.
>
> Clearly separate: 1. What the PR changes, 2. What is good, 3. Potential problems, 4. Questions for the author, 5. Manual tests to run, and 6. Approve, Comment, or Request changes.

Adapt the checklist to the type of change. For example, a database PR needs schema and data checks, while a container PR needs configuration and persistence checks.

## 9. After the review

Return to your own branch and refresh it when necessary:

```bash
git switch development
git pull --ff-only
```

A normal PR review checks one branch against the target branch. When several parallel branches will change shared architecture or configuration, use the separate `PR_Cross_review` notebook to assess their compatibility before merging.